# Step-by-step Sentinel-2 extraction 

This notebook runs the full AEREO pipeline — **Search**, **Prepare** and **Extract** — step by step. 
We query a STAC catalog for Sentinel-2 assets, build extraction tasks, and run each task through the orchestrator.

Each of these functions can be replaced by ANY function that conforms to the inputs and outputs of each step as follows:

| Step | Input | Output |
|------|-------|--------|
| Search | `(collections, intersects, start_datetime, end_datetime, **kwargs)` | `GeoDataFrame[AssetSchema]` |
| Build | `(GeoDataFrame[AssetSchema], ExtractionJob, **kwargs)` | `Sequence[ExtractionTask]` |
| Read | `(files: list[str], **kwargs)` | `xr.Dataset` |
| Write | `(ds: xr.Dataset, path: str, **kwargs)` | `str` |


# Configuring AOI, Grid and Extraction pipeline
Before we start, we must configure our `ExtractionJob`. 
This means defining the `grid` to use, the optional `resolution`/`margin` for reprojection, and the functions to use in the `extraction pipeline` — always conforming to the required inputs and outputs.

If we know all of this beforehand, we can run our pipeline anywhere by serializing the job, and define different pipelines using only config files!

![AEREO config interaction diagram](assets/aereo_configs.svg)


In [1]:
from aereo.builtins.search import search_stac
from datetime import datetime, timezone
from shapely.geometry import Polygon

from aereo.pipeline import ExtractionJob
from aereo.builtins import read_odc_stac, write_geotiff

# AOI polygon — Chocón reservoir, Argentina (inlined so the notebook has no file dependencies).
aoi_polygon = Polygon(
    [
        (-68.90986824592407, -39.23705421799603),
        (-68.65925870907353, -39.23705421799603),
        (-68.65925870907353, -39.41589522092947),
        (-68.90986824592407, -39.41589522092947),
        (-68.90986824592407, -39.23705421799603),
    ]
)

grid_dist = 10_000

job = ExtractionJob(
    name="sentinel2_sample",
    grid_dist=grid_dist,
    output_uri="/tmp/aereo_extraction",
    read=read_odc_stac,
    write=write_geotiff,
    target_aoi=aoi_polygon,
)

## Step 1 — Search: `search_stac`

The search provider queries the STAC API and returns a GeoDataFrame of matched assets. Each row corresponds to one requested asset (e.g. `red`, `nir`) from one STAC item. All parameters are passed explicitly.


In [2]:
assets = search_stac(
    stac_api_url="https://earth-search.aws.element84.com/v1",
    collections={"sentinel-2-l2a": ["red", "nir"]},
    intersects=aoi_polygon,
    start_datetime=datetime(2024, 1, 1, tzinfo=timezone.utc),
    end_datetime=datetime(2024, 1, 10, tzinfo=timezone.utc),
)

print(f"\u2713 Found {len(assets)} asset rows")
print("Columns:", list(assets.columns))
print("First rows:")
assets[["id", "collection", "channel_id", "crs", "href"]].head()

✓ Found 12 asset rows
Columns: ['id', 'collection', 'geometry', 'start_time', 'end_time', 'href', 'channel_id', 'crs', 'stac_item']
First rows:


,id,collection,channel_id,crs,href
0,S2A_19HDS_20240107_0_L2A_red,sentinel-2-l2a,red,EPSG:32719,https://sentinel-cogs.s3.us-west-2.amazonaws.c...
1,S2A_19HDS_20240107_0_L2A_nir,sentinel-2-l2a,nir,EPSG:32719,https://sentinel-cogs.s3.us-west-2.amazonaws.c...
2,S2A_19HES_20240107_0_L2A_red,sentinel-2-l2a,red,EPSG:32719,https://sentinel-cogs.s3.us-west-2.amazonaws.c...
3,S2A_19HES_20240107_0_L2A_nir,sentinel-2-l2a,nir,EPSG:32719,https://sentinel-cogs.s3.us-west-2.amazonaws.c...
4,S2B_19HDS_20240105_0_L2A_red,sentinel-2-l2a,red,EPSG:32719,https://sentinel-cogs.s3.us-west-2.amazonaws.c...


## Step 2 — Build tasks: `build_grouped_tasks`

The task builder takes the search-result GeoDataFrame and the `ExtractionJob` and produces a list of `ExtractionTask` objects. Tasks are grouped by start time and native CRS, and chunked by `cells_per_task`.


In [ ]:
from aereo.builtins.task_builder import build_grouped_tasks

tasks = build_grouped_tasks(
    search_results=assets,
    job=job,
    cells_per_task=4,
)
print(f"\u2713 Built {len(tasks)} extraction task(s)")
for i, task in enumerate(tasks):
    print(f"Task {i}: {task}")

✓ Built 6 extraction task(s)
Task 0: ExtractionTask(id='sentinel2_sample_2024-01-02 14:33:47.691000_EPSG:32719_0', n_assets=2, read=True, write=True, output_uri='/tmp/aereo_extraction')
Task 1: ExtractionTask(id='sentinel2_sample_2024-01-02 14:33:51.276000_EPSG:32719_1', n_assets=2, read=True, write=True, output_uri='/tmp/aereo_extraction')
Task 2: ExtractionTask(id='sentinel2_sample_2024-01-05 14:43:42.172000_EPSG:32719_2', n_assets=2, read=True, write=True, output_uri='/tmp/aereo_extraction')
Task 3: ExtractionTask(id='sentinel2_sample_2024-01-05 14:43:46.672000_EPSG:32719_3', n_assets=2, read=True, write=True, output_uri='/tmp/aereo_extraction')
Task 4: ExtractionTask(id='sentinel2_sample_2024-01-07 14:33:42.975000_EPSG:32719_4', n_assets=2, read=True, write=True, output_uri='/tmp/aereo_extraction')
Task 5: ExtractionTask(id='sentinel2_sample_2024-01-07 14:33:46.552000_EPSG:32719_5', n_assets=2, read=True, write=True, output_uri='/tmp/aereo_extraction')


In [4]:
# Inspect the first task in detail
task = tasks[0]

print("Task context:", dict(task.task_context))
print(f"Number of asset rows in task: {len(task.assets)}")

# Extraction stages available on the task (delegated from the job)
print("Extraction stages:")
print("  read callable:", callable(task.job.read))
print("  write callable:", callable(task.job.write))

Task context: {}
Number of asset rows in task: 2
Extraction stages:
  read callable: True
  write callable: True


## Step 3 — Read: `read_odc_stac`

The reader reconstructs `pystac.Item` objects from the asset table and uses `odc.stac.load` to build a lazy `xarray.Dataset` in the native CRS of the STAC items.


In [ ]:
from aereo.builtins.read import read_odc_stac

files = task.assets["href"].tolist()
ds_native = read_odc_stac(files, assets=task.assets)

print("Native dataset:")
ds_native
# ds_native["red"].plot()

Native dataset:


: 

## Step 4 — Write: `write_geotiff`

The writer serialises a dataset to a GeoTIFF at the path supplied by the caller and returns the written path.


In [ ]:
from pathlib import Path
from aereo.builtins.write import write_geotiff

# Write the native dataset to a temporary path
out_path = Path(job.output_uri) / "demo_native_red.tif"
written = write_geotiff(ds=ds_native, path=out_path)

print(f"\u2713 Wrote {written}")

In [ ]:
import rioxarray

rioxarray.open_rasterio(written)[0].plot()

## Putting it together: `run_task`

`run_task(task)` executes the fixed pipeline `read → preprocess → reproject → postprocess → write` for a single `ExtractionTask`. It is what `LocalExecutor` uses under the hood.


In [ ]:
from aereo.execution import run_task

artifacts_gdf = run_task(task)

print(f"\u2713 Single task produced {len(artifacts_gdf)} artifact row(s)")
artifacts_gdf[["id", "uri", "grid_cell", "geometry"]].head()

In [ ]:
rioxarray.open_rasterio(artifacts_gdf.iloc[3].uri)[0].plot()

## Batch execution with an executor

To run many tasks in parallel, pass the task list to an `Executor`. For COG/I/O-bound extractors like this one, `LocalExecutor(use_threads=True)` is the safer choice in Jupyter and avoids the pickling issues that can make `ProcessPoolExecutor` hang.


In [ ]:
from aereo.executors import LocalExecutor

executor = LocalExecutor(workers=4, use_threads=True)
artifacts = executor(tasks)

print(f"\u2713 Extracted {len(artifacts)} artifact row(s) from {len(tasks)} task(s)")
artifacts[["id", "uri", "grid_cell", "start_time"]].head()

In [ ]:
from aereo.viz import plot_artifact_patches

plot_artifact_patches(artifacts, ds_factor=10, cmap="viridis")